# Practice - full flow on gym_members.csv

A gym wants to know which members will renew their membership.

Do all the steps we did in class, on this new file. We want to predict `renewed`.

**Expected accuracy: about 0.82**

In [45]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

---
## Step 1 - Load and look

Expected shape: `(500, 10)`

In [46]:
df=pd.read_csv('gym_members.csv')
df.shape

(500, 10)

In [47]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   member_id          500 non-null    str    
 1   age                500 non-null    float64
 2   plan_type          500 non-null    str    
 3   workouts_per_week  500 non-null    int64  
 4   session_minutes    442 non-null    float64
 5   distance_km        479 non-null    float64
 6   months_active      491 non-null    float64
 7   trainer            500 non-null    str    
 8   referral_source    500 non-null    str    
 9   renewed            500 non-null    str    
dtypes: float64(4), int64(1), str(5)
memory usage: 39.2 KB


In [48]:
df.isnull().sum()

member_id             0
age                   0
plan_type             0
workouts_per_week     0
session_minutes      58
distance_km          21
months_active         9
trainer               0
referral_source       0
renewed               0
dtype: int64

---
## Step 2 - Separate numeric and categorical columns

Expected: 5 numeric, 3 categorical. `member_id` and `renewed` go in neither list.

In [49]:
nume_cols = ['age', 'workouts_per_week', 'session_minutes', 'distance_km', 'months_active']
cat_cols = ['plan_type', 'trainer', 'referral_source']

---
## Step 3 - Split into X and y

Expected: X `(500, 8)`, y `(500,)`

In [50]:

x = df.drop(columns=['member_id', 'renewed'])
y = df['renewed']

In [51]:
print(X.shape)

(500, 8)


In [52]:
print(y.shape)

(500,)


---
## Step 4 - Handle missing values

Fill every numeric column with its own mean.

Expected before: `session_minutes` 58, `distance_km` 21, `months_active` 9. After: all zero.

In [53]:

df[['session_minutes','distance_km','months_active']]=df[['session_minutes','distance_km','months_active']].fillna(df[['session_minutes','distance_km','months_active']].mean())
df.isnull().sum()

member_id            0
age                  0
plan_type            0
workouts_per_week    0
session_minutes      0
distance_km          0
months_active        0
trainer              0
referral_source      0
renewed              0
dtype: int64

---
## Step 5 - Encode and scale together

Use one `ColumnTransformer`.

Expected shape: `(500, 14)`

In [54]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
ct = ColumnTransformer(transformers=[('num', StandardScaler(), nume_cols),('cat', OneHotEncoder(), cat_cols)],remainder='passthrough')




In [55]:
X_transformed = ct.fit_transform(x)
print(X_transformed.shape)

(500, 14)


---
## Step 6 - Train and test split

Expected: train `(400, 14)`, test `(100, 14)`

In [56]:
x_train, x_test, y_train, y_test = train_test_split(
    X_transformed, y, test_size=0.2, random_state=42
)

In [57]:
print(type(X_transformed))
print(X_transformed.shape)
print(x_train.shape)
print(x_test.shape)

<class 'numpy.ndarray'>
(500, 14)
(400, 14)
(100, 14)


---
## Step 7 - Train the model

`KNeighborsClassifier` with `n_neighbors=5`, then predict on the test set.

In [59]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(x_train, y_train)
y_pred = knn.predict(x_test)

ValueError: Input X contains NaN.
KNeighborsClassifier does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

---
## Step 8 - Score

Print the accuracy of your model.

---
## Step 9 - Predict for one new member

Build a one row DataFrame with your own values and predict whether that member renews.

Use `transform`, not `fit_transform`.